# TextGrid-based vowel feature editing with ProMoNet

Select a neutral recording and an emotional donor recording. This notebook uses the manually corrected `phones` tiers in `audio/textgrids/*.TextGrid` to locate corresponding vowels, transfers donor **pitch**, **loudness**, and **periodicity**, and synthesizes the result with ProMoNet.

Important design choices:

- Whisper transcription and PPG decoding are **not** used for vowel labels or boundaries.
- Donor pitch, loudness, periodicity, duration, and experimental PPG are independently selectable. Donor PPG is extracted only when selected.
- All donor features start unchecked. Select Duration to use donor vowel length; otherwise neutral timing is preserved.
- A taper of `0 ms` performs a hard replacement; larger values blend each vowel edge.

## 1. Imports and environment

The notebook uses the existing `Python (ProMoNet WSL)` kernel. If an import below fails, update the environment from the project root:

```bash
conda env update -f environment.yml --prune
```

In [ ]:
from datetime import datetime
from pathlib import Path
import math
import re
import traceback

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from praatio import textgrid
import torch
import torch.nn.functional as F
import torchaudio

import promonet

from IPython.display import Audio, clear_output, display

GPU = 0 if torch.cuda.is_available() else None
print("ProMoNet sample rate:", promonet.SAMPLE_RATE)
print("ProMoNet hop size:", promonet.HOPSIZE)
print("GPU argument:", GPU)
if GPU is not None:
    print("GPU:", torch.cuda.get_device_name(GPU))

## 2. Paths, TextGrid parsing, and vowel matching

In [ ]:
ARPABET_VOWELS = {
    "AA", "AE", "AH", "AO", "AW", "AY", "EH", "ER",
    "EY", "IH", "IY", "OW", "OY", "UH", "UW",
}


def find_project_root(start=None):
    '''Find the nearest parent containing audio/textgrids.'''
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / "audio" / "textgrids").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find a project directory containing audio/textgrids. "
        f"Started at {start}."
    )


def discover_audio_textgrid_pairs(audio_dir, textgrid_dir):
    '''Return WAV/TextGrid pairs keyed by the WAV stem.'''
    audio_dir = Path(audio_dir)
    textgrid_dir = Path(textgrid_dir)
    textgrids = {
        path.stem.casefold(): path
        for path in textgrid_dir.glob("*.TextGrid")
    }
    textgrids.update({
        path.stem.casefold(): path
        for path in textgrid_dir.glob("*.textgrid")
    })
    pairs = {}
    for audio_path in sorted(audio_dir.glob("*.wav")):
        textgrid_path = textgrids.get(audio_path.stem.casefold())
        if textgrid_path is not None:
            pairs[audio_path.stem] = {
                "audio": audio_path,
                "textgrid": textgrid_path,
            }
    if not pairs:
        raise FileNotFoundError(
            f"No same-stem WAV/TextGrid pairs found in {audio_dir} and {textgrid_dir}."
        )
    return pairs


def normalize_arpabet_phone(label):
    '''Uppercase an ARPAbet phone and remove a trailing stress number.'''
    return re.sub(r"[012]+$", "", str(label).strip().upper())


def _tier_name(textgrid_object, requested_name):
    matches = {
        name.casefold(): name
        for name in textgrid_object.tierNames
    }
    resolved = matches.get(requested_name.casefold())
    if resolved is None:
        raise ValueError(
            f"Required tier {requested_name!r} is missing. "
            f"Available tiers: {list(textgrid_object.tierNames)}"
        )
    return resolved


def _word_for_interval(start_s, end_s, word_entries):
    best_word = ""
    best_overlap = 0.0
    for entry in word_entries:
        label = str(entry.label).strip()
        if not label:
            continue
        overlap = max(0.0, min(end_s, entry.end) - max(start_s, entry.start))
        if overlap > best_overlap:
            best_word = label
            best_overlap = overlap
    return best_word


def read_vowel_intervals(textgrid_path, phone_tier="phones", word_tier="words"):
    '''Read vowel intervals from a TextGrid into a one-row-per-vowel table.'''
    textgrid_path = Path(textgrid_path)
    if not textgrid_path.is_file():
        raise FileNotFoundError(f"TextGrid does not exist: {textgrid_path}")

    tg = textgrid.openTextgrid(str(textgrid_path), includeEmptyIntervals=True)
    phones = tg.getTier(_tier_name(tg, phone_tier)).entries
    try:
        words = tg.getTier(_tier_name(tg, word_tier)).entries
    except ValueError:
        words = []

    rows = []
    for entry in phones:
        original_phone = str(entry.label).strip()
        normalized_phone = normalize_arpabet_phone(original_phone)
        if normalized_phone not in ARPABET_VOWELS:
            continue
        rows.append({
            "vowel_id": len(rows) + 1,
            "phone": original_phone,
            "normalized_phone": normalized_phone,
            "word": _word_for_interval(entry.start, entry.end, words),
            "start_s": float(entry.start),
            "end_s": float(entry.end),
            "duration_s": float(entry.end - entry.start),
        })

    if not rows:
        raise ValueError(
            f"No ARPAbet vowels were found in tier {phone_tier!r}: {textgrid_path}"
        )
    return pd.DataFrame(rows)


def seconds_to_frame_span(start_s, end_s, number_of_frames):
    '''Convert a positive half-open time interval to clamped feature frames.'''
    if not (math.isfinite(start_s) and math.isfinite(end_s)):
        raise ValueError("TextGrid interval times must be finite.")
    if start_s < 0 or end_s <= start_s:
        raise ValueError(f"Invalid TextGrid interval: [{start_s}, {end_s})")
    if number_of_frames <= 0:
        raise ValueError("number_of_frames must be positive.")

    frames_per_second = promonet.SAMPLE_RATE / promonet.HOPSIZE
    # The same timestamp must map to the same boundary frame for adjacent phones.
    start_frame = round(start_s * frames_per_second)
    end_frame = round(end_s * frames_per_second)
    start_frame = min(max(start_frame, 0), number_of_frames)
    end_frame = min(max(end_frame, 0), number_of_frames)
    if end_frame <= start_frame:
        raise ValueError(
            "TextGrid interval collapses after conversion to ProMoNet frames: "
            f"[{start_s:.6f}, {end_s:.6f}) -> [{start_frame}, {end_frame})"
        )
    return start_frame, end_frame


def match_vowel_sequences(
    neutral_vowels,
    donor_vowels,
    neutral_frames=None,
    donor_frames=None,
):
    '''Pair vowels by order and normalized ARPAbet label.'''
    neutral_sequence = neutral_vowels["normalized_phone"].tolist()
    donor_sequence = donor_vowels["normalized_phone"].tolist()
    if neutral_sequence != donor_sequence:
        maximum = max(len(neutral_vowels), len(donor_vowels))
        comparison = pd.DataFrame({
            "vowel_id": range(1, maximum + 1),
            "neutral": neutral_sequence + [None] * (maximum - len(neutral_sequence)),
            "donor": donor_sequence + [None] * (maximum - len(donor_sequence)),
        })
        raise ValueError(
            "Neutral and donor vowel sequences do not match. "
            "Correct the TextGrids before editing.\n\n"
            + comparison.to_string(index=False)
        )

    rows = []
    for neutral_row, donor_row in zip(
        neutral_vowels.to_dict("records"),
        donor_vowels.to_dict("records"),
    ):
        row = {
            "vowel_id": int(neutral_row["vowel_id"]),
            "normalized_phone": neutral_row["normalized_phone"],
            "neutral_phone": neutral_row["phone"],
            "donor_phone": donor_row["phone"],
            "word": neutral_row["word"],
            "neutral_start_s": neutral_row["start_s"],
            "neutral_end_s": neutral_row["end_s"],
            "neutral_duration_s": neutral_row["duration_s"],
            "donor_start_s": donor_row["start_s"],
            "donor_end_s": donor_row["end_s"],
            "donor_duration_s": donor_row["duration_s"],
        }
        if neutral_frames is not None and donor_frames is not None:
            neutral_start, neutral_end = seconds_to_frame_span(
                neutral_row["start_s"], neutral_row["end_s"], neutral_frames
            )
            donor_start, donor_end = seconds_to_frame_span(
                donor_row["start_s"], donor_row["end_s"], donor_frames
            )
            row.update({
                "neutral_start_frame": neutral_start,
                "neutral_end_frame": neutral_end,
                "donor_start_frame": donor_start,
                "donor_end_frame": donor_end,
            })
        rows.append(row)

    matched = pd.DataFrame(rows)
    if neutral_frames is not None:
        neutral_starts = matched["neutral_start_frame"].tolist()
        neutral_ends = matched["neutral_end_frame"].tolist()
        if any(start < previous_end for start, previous_end in zip(neutral_starts[1:], neutral_ends[:-1])):
            raise ValueError("Neutral vowel frame spans overlap after time conversion.")
    return matched


PROJECT_ROOT = find_project_root()
AUDIO_DIR = PROJECT_ROOT / "audio"
TEXTGRID_DIR = AUDIO_DIR / "textgrids"
OUTPUT_DIR = PROJECT_ROOT / "vowel_edit_pipeline" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_TEXTGRID_PAIRS = discover_audio_textgrid_pairs(AUDIO_DIR, TEXTGRID_DIR)

print("Project root:", PROJECT_ROOT)
print("Matched recordings:", len(AUDIO_TEXTGRID_PAIRS))
for stem in AUDIO_TEXTGRID_PAIRS:
    print(" -", stem)

## 3. ProMoNet feature extraction and trajectory transfer

In [ ]:
ACOUSTIC_FEATURES = ("loudness", "pitch", "periodicity")
TRANSFER_FEATURE_ORDER = ("pitch", "loudness", "periodicity", "duration", "ppg")
TRANSFERABLE_FEATURES = frozenset(TRANSFER_FEATURE_ORDER)
SYNTHESIS_FEATURES = (*ACOUSTIC_FEATURES, "ppg")


def extract_promonet_features(audio_path, include_ppg, gpu=GPU):
    '''Load audio and extract only the features needed for this pipeline.'''
    audio_path = Path(audio_path)
    if not audio_path.is_file():
        raise FileNotFoundError(f"Audio does not exist: {audio_path}")
    requested = list(ACOUSTIC_FEATURES)
    if include_ppg:
        requested.append("ppg")

    audio = promonet.load.audio(audio_path)
    with torch.inference_mode():
        returned = promonet.preprocess.from_audio(
            audio,
            features=requested,
            gpu=gpu,
        )
    features = dict(zip(requested, returned))
    frame_counts = {name: value.shape[-1] for name, value in features.items()}
    if len(set(frame_counts.values())) != 1:
        raise ValueError(f"Feature frame counts do not agree: {frame_counts}")
    return audio, features


def resample_time_axis(trajectory, target_frames):
    '''Linearly resize the last (time) axis and preserve leading dimensions.'''
    if target_frames <= 0:
        raise ValueError("target_frames must be positive.")
    source_frames = trajectory.shape[-1]
    if source_frames <= 0:
        raise ValueError("Cannot resample an empty trajectory.")
    if source_frames == target_frames:
        return trajectory.clone()

    original_shape = trajectory.shape
    flat = trajectory.reshape(-1, 1, source_frames)
    resized = F.interpolate(
        flat.float(),
        size=target_frames,
        mode="linear",
        align_corners=False,
    )
    return resized.to(dtype=trajectory.dtype).reshape(*original_shape[:-1], target_frames)


def normalize_ppg(ppg):
    '''Clamp numerical noise and normalize probabilities across phone channels.'''
    ppg = ppg.clamp_min(0.0)
    return ppg / ppg.sum(dim=-2, keepdim=True).clamp_min(1e-8)


def make_boundary_taper(
    frame_count,
    taper_ms,
    sample_rate=promonet.SAMPLE_RATE,
    hop_size=promonet.HOPSIZE,
    device=None,
    dtype=torch.float32,
):
    '''Create acoustic donor weights; 0 ms is a hard replacement.'''
    if frame_count <= 0:
        raise ValueError("frame_count must be positive.")
    if taper_ms < 0:
        raise ValueError("taper_ms cannot be negative.")
    weights = torch.ones(frame_count, device=device, dtype=dtype)
    if taper_ms == 0:
        return weights

    requested_frames = round((taper_ms / 1000.0) * sample_rate / hop_size)
    taper_frames = min(max(requested_frames, 0), frame_count // 2)
    if taper_frames == 0:
        return weights
    ramp = torch.arange(taper_frames, device=device, dtype=dtype) / taper_frames
    weights[:taper_frames] = ramp
    weights[-taper_frames:] = ramp.flip(0)
    return weights


def _blend_trajectories(neutral_reference, donor_trajectory, donor_weights):
    view_shape = (1,) * (donor_trajectory.ndim - 1) + (donor_weights.numel(),)
    weights = donor_weights.reshape(view_shape).to(
        device=donor_trajectory.device,
        dtype=donor_trajectory.dtype,
    )
    return neutral_reference * (1.0 - weights) + donor_trajectory * weights


def validate_transferred_features(transferred_features):
    '''Return a validated feature set in canonical display order.'''
    selected = {str(name).strip().lower() for name in transferred_features}
    if not selected:
        raise ValueError("Select at least one donor feature to transfer.")
    unknown = selected - TRANSFERABLE_FEATURES
    if unknown:
        raise ValueError(f"Unknown donor features: {sorted(unknown)}")
    return tuple(name for name in TRANSFER_FEATURE_ORDER if name in selected)


def transfer_vowel_features(
    neutral_features,
    donor_features,
    matched_vowels,
    selected_vowel_ids,
    transferred_features,
    taper_ms=35.0,
):
    '''Rebuild neutral features using only the selected donor trajectories.'''
    transferred_features = validate_transferred_features(transferred_features)
    transferred_set = set(transferred_features)
    selected_vowel_ids = {int(value) for value in selected_vowel_ids}
    if not selected_vowel_ids:
        raise ValueError("Select at least one vowel to edit.")

    available_ids = set(matched_vowels["vowel_id"].astype(int))
    unknown_ids = selected_vowel_ids - available_ids
    if unknown_ids:
        raise ValueError(f"Unknown vowel IDs: {sorted(unknown_ids)}")
    for name in SYNTHESIS_FEATURES:
        if name not in neutral_features:
            raise ValueError(f"Neutral feature is missing: {name}")
    for name in ACOUSTIC_FEATURES:
        if name not in donor_features:
            raise ValueError(f"Donor feature is missing: {name}")
    if "ppg" in transferred_set and "ppg" not in donor_features:
        raise ValueError("Donor PPG was selected but was not extracted.")

    donor_duration = "duration" in transferred_set
    selected_acoustics = transferred_set & set(ACOUSTIC_FEATURES)
    neutral_total_frames = neutral_features["pitch"].shape[-1]
    chunks = {name: [] for name in SYNTHESIS_FEATURES}
    audit_rows = []
    neutral_cursor = 0
    output_cursor = 0

    for row in matched_vowels.to_dict("records"):
        vowel_id = int(row["vowel_id"])
        neutral_start = int(row["neutral_start_frame"])
        neutral_end = int(row["neutral_end_frame"])
        donor_start = int(row["donor_start_frame"])
        donor_end = int(row["donor_end_frame"])
        if neutral_start < neutral_cursor:
            raise ValueError("Neutral vowel frame spans must be ordered and non-overlapping.")

        gap_frames = neutral_start - neutral_cursor
        for name in SYNTHESIS_FEATURES:
            chunks[name].append(neutral_features[name][..., neutral_cursor:neutral_start])
        output_cursor += gap_frames

        neutral_length = neutral_end - neutral_start
        donor_length = donor_end - donor_start
        selected = vowel_id in selected_vowel_ids
        output_start = output_cursor

        if selected:
            output_length = donor_length if donor_duration else neutral_length
            donor_weights = make_boundary_taper(
                output_length,
                taper_ms,
                device=neutral_features["pitch"].device,
                dtype=neutral_features["pitch"].dtype,
            )
            for name in ACOUSTIC_FEATURES:
                neutral_segment = resample_time_axis(
                    neutral_features[name][..., neutral_start:neutral_end],
                    output_length,
                )
                if name in selected_acoustics:
                    donor_segment = resample_time_axis(
                        donor_features[name][..., donor_start:donor_end],
                        output_length,
                    )
                    output_segment = _blend_trajectories(
                        neutral_segment, donor_segment, donor_weights
                    )
                else:
                    output_segment = neutral_segment
                chunks[name].append(output_segment)

            if "ppg" in transferred_set:
                # PPG intentionally uses a hard splice; acoustic taper is not applied.
                output_ppg = resample_time_axis(
                    donor_features["ppg"][..., donor_start:donor_end],
                    output_length,
                )
                output_ppg = normalize_ppg(output_ppg)
            else:
                neutral_ppg = neutral_features["ppg"][..., neutral_start:neutral_end]
                if output_length == neutral_length:
                    output_ppg = neutral_ppg.clone()
                else:
                    output_ppg = normalize_ppg(
                        resample_time_axis(neutral_ppg, output_length)
                    )
            chunks["ppg"].append(output_ppg)
            source = "donor: " + ",".join(transferred_features)
        else:
            output_length = neutral_length
            for name in SYNTHESIS_FEATURES:
                chunks[name].append(neutral_features[name][..., neutral_start:neutral_end])
            source = "neutral (unchanged)"

        output_end = output_start + output_length
        audit_rows.append({
            "vowel_id": vowel_id,
            "phone": row["neutral_phone"],
            "normalized_phone": row["normalized_phone"],
            "word": row["word"],
            "selected": selected,
            "source": source,
            "duration_mode": "donor" if donor_duration else "neutral",
            "taper_ms": float(taper_ms) if selected and selected_acoustics else 0.0,
            "neutral_start_s": row["neutral_start_s"],
            "neutral_end_s": row["neutral_end_s"],
            "donor_start_s": row["donor_start_s"],
            "donor_end_s": row["donor_end_s"],
            "neutral_start_frame": neutral_start,
            "neutral_end_frame": neutral_end,
            "donor_start_frame": donor_start,
            "donor_end_frame": donor_end,
            "output_start_frame": output_start,
            "output_end_frame": output_end,
            "neutral_frames": neutral_length,
            "donor_frames": donor_length,
            "output_frames": output_length,
            "output_duration_s": output_length * promonet.HOPSIZE / promonet.SAMPLE_RATE,
            "transferred_features": ",".join(transferred_features) if selected else "",
        })
        output_cursor = output_end
        neutral_cursor = neutral_end

    for name in SYNTHESIS_FEATURES:
        chunks[name].append(neutral_features[name][..., neutral_cursor:neutral_total_frames])
    edited = {name: torch.cat(chunks[name], dim=-1) for name in SYNTHESIS_FEATURES}
    audit = pd.DataFrame(audit_rows)
    validate_edited_features(edited)
    return edited, audit


def validate_edited_features(features):
    '''Validate the tensor contract required by ProMoNet synthesis.'''
    missing = set(SYNTHESIS_FEATURES) - set(features)
    if missing:
        raise ValueError(f"Missing synthesis features: {sorted(missing)}")
    lengths = {name: features[name].shape[-1] for name in SYNTHESIS_FEATURES}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"Edited feature lengths do not match: {lengths}")
    for name in SYNTHESIS_FEATURES:
        if not torch.isfinite(features[name]).all():
            raise ValueError(f"Feature contains non-finite values: {name}")
    periodicity = features["periodicity"]
    if periodicity.min() < -1e-5 or periodicity.max() > 1.0 + 1e-5:
        raise ValueError("Periodicity must stay within [0, 1].")
    ppg = features["ppg"]
    if ppg.ndim != 3:
        raise ValueError(f"Expected PPG shape [batch, phones, time], got {tuple(ppg.shape)}")
    ppg_sums = ppg.sum(dim=-2)
    if not torch.allclose(ppg_sums, torch.ones_like(ppg_sums), atol=2e-3, rtol=2e-3):
        raise ValueError("PPG probabilities are not normalized across phone channels.")
    return True

## 4. Plotting, synthesis, and artifact saving

In [ ]:
def _global_loudness(loudness):
    return promonet.preprocess.loudness.band_average(loudness, 1).detach().cpu().reshape(-1)


def _time_axis(number_of_frames):
    return torch.arange(number_of_frames).numpy() * promonet.HOPSIZE / promonet.SAMPLE_RATE


def plot_vowel_edit(
    neutral_features,
    donor_features,
    edited_features,
    audit,
    transferred_features,
):
    '''Plot complete neutral, donor, and edited acoustic trajectories.'''
    transferred_features = validate_transferred_features(transferred_features)
    plot_values = {
        "Pitch (Hz)": (
            neutral_features["pitch"].detach().cpu().reshape(-1),
            donor_features["pitch"].detach().cpu().reshape(-1),
            edited_features["pitch"].detach().cpu().reshape(-1),
        ),
        "Global loudness (dB)": (
            _global_loudness(neutral_features["loudness"]),
            _global_loudness(donor_features["loudness"]),
            _global_loudness(edited_features["loudness"]),
        ),
        "Periodicity": (
            neutral_features["periodicity"].detach().cpu().reshape(-1),
            donor_features["periodicity"].detach().cpu().reshape(-1),
            edited_features["periodicity"].detach().cpu().reshape(-1),
        ),
    }
    figure, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=False)
    for axis, (label, (neutral, donor, edited)) in zip(axes, plot_values.items()):
        axis.plot(_time_axis(len(neutral)), neutral, label="neutral", alpha=0.75)
        axis.plot(_time_axis(len(donor)), donor, label="donor", alpha=0.65)
        axis.plot(_time_axis(len(edited)), edited, label="edited", linewidth=1.8)
        for row in audit.loc[audit["selected"]].to_dict("records"):
            start = row["output_start_frame"] * promonet.HOPSIZE / promonet.SAMPLE_RATE
            end = row["output_end_frame"] * promonet.HOPSIZE / promonet.SAMPLE_RATE
            axis.axvspan(start, end, color="tab:orange", alpha=0.10)
        axis.set_ylabel(label)
        axis.grid(alpha=0.2)
    axes[0].legend(ncol=3)
    axes[-1].set_xlabel("Time (seconds; each full trajectory uses its own timeline)")
    feature_label = ", ".join(transferred_features)
    figure.suptitle(f"TextGrid-edited trajectories — donor features: {feature_label}")
    figure.tight_layout()
    plt.show()
    return figure


def resolve_checkpoint(checkpoint_text):
    '''Return None for pretrained synthesis or validate a supplied checkpoint path.'''
    checkpoint_text = str(checkpoint_text).strip()
    if not checkpoint_text:
        return None
    checkpoint = Path(checkpoint_text).expanduser()
    if not checkpoint.exists():
        raise FileNotFoundError(f"Checkpoint path does not exist: {checkpoint}")
    return checkpoint


def synthesize_features(features, speaker_id=0, checkpoint=None, gpu=GPU):
    validate_edited_features(features)
    with torch.inference_mode():
        return promonet.synthesize.from_features(
            features["loudness"],
            features["pitch"],
            features["periodicity"],
            features["ppg"],
            speaker=int(speaker_id),
            checkpoint=checkpoint,
            gpu=gpu,
        )


def save_run_artifacts(
    edited_audio,
    edited_features,
    audit,
    neutral_path,
    donor_path,
    transferred_features,
    taper_ms,
    speaker_id,
    checkpoint,
    output_dir=OUTPUT_DIR,
):
    '''Save WAV, per-vowel CSV, and CPU feature tensors under one prefix.'''
    transferred_features = validate_transferred_features(transferred_features)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    feature_slug = "-".join(transferred_features)
    prefix = output_dir / (
        f"{Path(neutral_path).stem}__from__{Path(donor_path).stem}__"
        f"features-{feature_slug}__{timestamp}"
    )
    wav_path = prefix.with_suffix(".wav")
    csv_path = Path(f"{prefix}_audit.csv")
    tensor_path = Path(f"{prefix}_features.pt")

    torchaudio.save(wav_path, edited_audio.detach().cpu(), promonet.SAMPLE_RATE)
    audit.to_csv(csv_path, index=False)
    bundle = {
        "features": {
            name: value.detach().cpu()
            for name, value in edited_features.items()
        },
        "metadata": {
            "neutral_audio": str(Path(neutral_path).resolve()),
            "donor_audio": str(Path(donor_path).resolve()),
            "transferred_features": list(transferred_features),
            "duration_mode": "donor" if "duration" in transferred_features else "neutral",
            "taper_ms": float(taper_ms),
            "speaker_id": int(speaker_id),
            "checkpoint": None if checkpoint is None else str(Path(checkpoint).resolve()),
            "sample_rate": int(promonet.SAMPLE_RATE),
            "hop_size": int(promonet.HOPSIZE),
            "created_at": datetime.now().isoformat(timespec="seconds"),
        },
        "audit": audit.to_dict("records"),
    }
    torch.save(bundle, tensor_path)
    return {
        "wav": wav_path,
        "audit_csv": csv_path,
        "feature_tensors": tensor_path,
    }


def checkpoint_step(checkpoint_path):
    '''Parse the numerical step from a ProMoNet checkpoint filename.'''
    match = re.search(r"-(\d+)\.pt$", Path(checkpoint_path).name)
    if match is None:
        raise ValueError(f"Unrecognized checkpoint filename: {checkpoint_path}")
    return int(match.group(1))


def discover_adapted_models():
    '''Find the numerically latest adapted generator for every speaker.'''
    run_root = promonet.RUNS_DIR / promonet.CONFIG / "adapt"
    manifest_directory = PROJECT_ROOT / "vowel_edit_pipeline" / "adaptation_manifests"
    models = []
    if not run_root.is_dir():
        return models
    for speaker_directory in sorted(path for path in run_root.iterdir() if path.is_dir()):
        generators = list(speaker_directory.glob("generator-*.pt"))
        if not generators:
            continue
        latest = max(generators, key=checkpoint_step)
        models.append({
            "speaker_name": speaker_directory.name,
            "generator": latest,
            "step": checkpoint_step(latest),
            "adaptation_steps": max(0, checkpoint_step(latest) - int(promonet.STEPS)),
            "manifest_status": (
                "managed"
                if (manifest_directory / f"{speaker_directory.name}.json").is_file()
                else "legacy"
            ),
        })
    return models


def build_model_options():
    '''Build stable dropdown values for pretrained, adapted, and custom models.'''
    options = [("Pretrained model", ("pretrained", ""))]
    for model in discover_adapted_models():
        status = "" if model["manifest_status"] == "managed" else " (legacy)"
        label = (
            f"Adapted: {model['speaker_name']} — "
            f"{Path(model['generator']).name}{status}"
        )
        options.append((label, ("adapted", str(model["generator"]))))
    options.append(("Custom checkpoint path", ("custom", "")))
    return options

## 5. Lightweight self-checks

These checks use the current TextGrids and small synthetic tensors. They do not load a ProMoNet checkpoint or synthesize audio.

In [ ]:
def run_pipeline_self_checks():
    expected_vowels = ["AH", "IH", "OW", "AA", "AH", "IY"]
    for stem, pair in AUDIO_TEXTGRID_PAIRS.items():
        vowels = read_vowel_intervals(pair["textgrid"])
        observed = vowels["normalized_phone"].tolist()
        assert observed == expected_vowels, f"{stem}: {observed}"

    neutral = {
        "loudness": torch.arange(80, dtype=torch.float32).reshape(8, 10),
        "pitch": torch.arange(10, dtype=torch.float32).reshape(1, 10),
        "periodicity": torch.linspace(0.1, 0.9, 10).reshape(1, 10),
        "ppg": torch.stack((
            torch.full((10,), 0.7),
            torch.full((10,), 0.3),
        )).unsqueeze(0),
    }
    donor = {
        "loudness": torch.full((8, 12), 100.0),
        "pitch": torch.full((1, 12), 100.0),
        "periodicity": torch.full((1, 12), 0.8),
        "ppg": torch.stack((
            torch.full((12,), 0.2),
            torch.full((12,), 0.8),
        )).unsqueeze(0),
    }
    matches = pd.DataFrame([
        {
            "vowel_id": 1, "normalized_phone": "AA",
            "neutral_phone": "AA1", "donor_phone": "AA1", "word": "one",
            "neutral_start_s": 0.02, "neutral_end_s": 0.04,
            "donor_start_s": 0.01, "donor_end_s": 0.04,
            "neutral_start_frame": 2, "neutral_end_frame": 4,
            "donor_start_frame": 1, "donor_end_frame": 4,
        },
        {
            "vowel_id": 2, "normalized_phone": "IY",
            "neutral_phone": "IY1", "donor_phone": "IY1", "word": "two",
            "neutral_start_s": 0.06, "neutral_end_s": 0.08,
            "donor_start_s": 0.07, "donor_end_s": 0.09,
            "neutral_start_frame": 6, "neutral_end_frame": 8,
            "donor_start_frame": 7, "donor_end_frame": 9,
        },
    ])

    def transfer(features, taper_ms=0):
        return transfer_vowel_features(
            neutral, donor, matches, {1}, features, taper_ms=taper_ms
        )[0]

    pitch_only = transfer({"pitch"})
    assert torch.equal(pitch_only["pitch"][..., 2:4], torch.full((1, 2), 100.0))
    for name in ("loudness", "periodicity", "ppg"):
        assert torch.equal(pitch_only[name], neutral[name])

    loudness_only = transfer({"loudness"})
    assert torch.equal(loudness_only["loudness"][..., 2:4], torch.full((8, 2), 100.0))
    assert torch.equal(loudness_only["pitch"], neutral["pitch"])

    periodicity_only = transfer({"periodicity"})
    assert torch.equal(
        periodicity_only["periodicity"][..., 2:4], torch.full((1, 2), 0.8)
    )
    assert torch.equal(periodicity_only["pitch"], neutral["pitch"])

    ppg_only = transfer({"ppg"}, taper_ms=100)
    assert torch.allclose(ppg_only["ppg"][..., 2:4], donor["ppg"][..., :2])
    for name in ACOUSTIC_FEATURES:
        assert torch.equal(ppg_only[name], neutral[name])

    duration_only = transfer({"duration"})
    assert duration_only["pitch"].shape[-1] == 11
    assert torch.equal(duration_only["pitch"][..., :2], neutral["pitch"][..., :2])
    assert torch.equal(duration_only["pitch"][..., 5:], neutral["pitch"][..., 4:])

    pitch_loudness = transfer({"pitch", "loudness"})
    assert torch.equal(pitch_loudness["periodicity"], neutral["periodicity"])
    assert torch.equal(pitch_loudness["ppg"], neutral["ppg"])

    pitch_duration = transfer({"pitch", "duration"})
    assert pitch_duration["pitch"].shape[-1] == 11
    assert torch.equal(pitch_duration["pitch"][..., 2:5], torch.full((1, 3), 100.0))

    all_features = transfer(TRANSFERABLE_FEATURES)
    assert all_features["pitch"].shape[-1] == 11
    assert torch.allclose(all_features["ppg"][..., 2:5], donor["ppg"][..., 1:4])
    validate_edited_features(all_features)

    tapered = transfer({"pitch", "ppg"}, taper_ms=35)
    assert torch.equal(tapered["pitch"][..., 2], neutral["pitch"][..., 2])
    assert torch.allclose(tapered["ppg"][..., 2], donor["ppg"][..., 1])

    for invalid_features in (set(), {"harmonics"}):
        try:
            transfer_vowel_features(
                neutral, donor, matches, {1}, invalid_features, taper_ms=0
            )
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid feature selection was accepted: {invalid_features}")

    try:
        transfer_vowel_features(neutral, donor, matches, set(), {"pitch"})
    except ValueError as error:
        assert "Select at least one vowel" in str(error)
    else:
        raise AssertionError("Empty vowel selection should fail.")

    mismatched = read_vowel_intervals(
        next(iter(AUDIO_TEXTGRID_PAIRS.values()))["textgrid"]
    ).iloc[:-1]
    complete = read_vowel_intervals(next(iter(AUDIO_TEXTGRID_PAIRS.values()))["textgrid"])
    try:
        match_vowel_sequences(complete, mismatched)
    except ValueError as error:
        assert "do not match" in str(error)
    else:
        raise AssertionError("Mismatched vowel sequences should fail.")

    return {
        "textgrids_checked": len(AUDIO_TEXTGRID_PAIRS),
        "expected_vowels": expected_vowels,
        "individual_features_checked": list(TRANSFER_FEATURE_ORDER),
        "combination_checks": "passed",
        "ppg_hard_splice_check": "passed",
    }


SELF_CHECK_RESULTS = run_pipeline_self_checks()
display(SELF_CHECK_RESULTS)

## 6. Interactive editor

In [ ]:
def run_vowel_edit(
    neutral_stem,
    donor_stem,
    selected_vowel_ids,
    transferred_features,
    taper_ms,
    speaker_id,
    checkpoint_text,
    gpu=GPU,
):
    '''Run one complete TextGrid-guided selective donor transfer.'''
    transferred_features = validate_transferred_features(transferred_features)
    if neutral_stem == donor_stem:
        raise ValueError("Neutral and donor recordings must be different files.")
    neutral_pair = AUDIO_TEXTGRID_PAIRS[neutral_stem]
    donor_pair = AUDIO_TEXTGRID_PAIRS[donor_stem]
    checkpoint = resolve_checkpoint(checkpoint_text)

    neutral_vowels = read_vowel_intervals(neutral_pair["textgrid"])
    donor_vowels = read_vowel_intervals(donor_pair["textgrid"])
    match_vowel_sequences(neutral_vowels, donor_vowels)

    print("Extracting neutral acoustics and neutral PPG ...")
    neutral_audio, neutral_features = extract_promonet_features(
        neutral_pair["audio"], include_ppg=True, gpu=gpu
    )
    donor_uses_ppg = "ppg" in transferred_features
    print(
        "Extracting donor acoustics"
        + (" and experimental donor PPG ..." if donor_uses_ppg else " (no donor PPG) ...")
    )
    donor_audio, donor_features = extract_promonet_features(
        donor_pair["audio"], include_ppg=donor_uses_ppg, gpu=gpu
    )
    matches = match_vowel_sequences(
        neutral_vowels,
        donor_vowels,
        neutral_frames=neutral_features["pitch"].shape[-1],
        donor_frames=donor_features["pitch"].shape[-1],
    )
    edited_features, audit = transfer_vowel_features(
        neutral_features,
        donor_features,
        matches,
        selected_vowel_ids,
        transferred_features=transferred_features,
        taper_ms=taper_ms,
    )

    print("Synthesizing unedited neutral baseline ...")
    neutral_reconstruction = synthesize_features(
        neutral_features,
        speaker_id=speaker_id,
        checkpoint=checkpoint,
        gpu=gpu,
    )
    print("Synthesizing edited sentence ...")
    edited_audio = synthesize_features(
        edited_features,
        speaker_id=speaker_id,
        checkpoint=checkpoint,
        gpu=gpu,
    )
    paths = save_run_artifacts(
        edited_audio,
        edited_features,
        audit,
        neutral_pair["audio"],
        donor_pair["audio"],
        transferred_features,
        taper_ms,
        speaker_id,
        checkpoint,
    )

    display_columns = [
        "vowel_id", "phone", "word", "selected", "transferred_features",
        "neutral_frames", "donor_frames", "output_frames",
        "output_duration_s", "taper_ms",
    ]
    display(audit[display_columns])
    plot_vowel_edit(
        neutral_features,
        donor_features,
        edited_features,
        audit,
        transferred_features,
    )

    print("Original neutral")
    display(Audio(filename=str(neutral_pair["audio"])))
    print("Original donor")
    display(Audio(filename=str(donor_pair["audio"])))
    print("Unedited neutral reconstruction")
    display(Audio(neutral_reconstruction.detach().cpu(), rate=promonet.SAMPLE_RATE))
    print("Edited reconstruction")
    display(Audio(edited_audio.detach().cpu(), rate=promonet.SAMPLE_RATE))
    print("Saved artifacts:")
    for kind, artifact_path in paths.items():
        print(f" - {kind}: {artifact_path}")
    return {
        "edited_audio": edited_audio,
        "edited_features": edited_features,
        "audit": audit,
        "paths": paths,
    }


def build_notebook_controls():
    '''Create file, vowel, feature, taper, and model controls.'''
    stems = list(AUDIO_TEXTGRID_PAIRS)
    neutral_default = next(
        (stem for stem in stems if re.search(r"(^|_)neu($|_)", stem, re.I)),
        stems[0],
    )
    donor_default = next(stem for stem in stems if stem != neutral_default)

    neutral_dropdown = widgets.Dropdown(
        options=stems,
        value=neutral_default,
        description="Neutral:",
        layout=widgets.Layout(width="500px"),
    )
    donor_dropdown = widgets.Dropdown(
        options=stems,
        value=donor_default,
        description="Donor:",
        layout=widgets.Layout(width="500px"),
    )
    vowel_select = widgets.SelectMultiple(
        options=[],
        description="Vowels:",
        rows=8,
        layout=widgets.Layout(width="700px"),
    )
    select_all_button = widgets.Button(description="Select all vowels")
    clear_button = widgets.Button(description="Clear selection")
    feature_checkboxes = {
        "pitch": widgets.Checkbox(value=False, description="Pitch", indent=False),
        "loudness": widgets.Checkbox(value=False, description="Loudness", indent=False),
        "periodicity": widgets.Checkbox(value=False, description="Periodicity", indent=False),
        "duration": widgets.Checkbox(value=False, description="Duration", indent=False),
        "ppg": widgets.Checkbox(value=False, description="PPG (experimental)", indent=False),
    }
    feature_status = widgets.HTML(
        value="<span style='color:#b33'><b>Select at least one donor feature.</b></span>"
    )
    taper_slider = widgets.FloatSlider(
        value=35.0,
        min=0.0,
        max=100.0,
        step=5.0,
        description="Acoustic taper (ms):",
        continuous_update=False,
        readout_format=".0f",
        disabled=True,
        layout=widgets.Layout(width="650px"),
    )
    model_dropdown = widgets.Dropdown(
        options=build_model_options(),
        value=("pretrained", ""),
        description="Model:",
        layout=widgets.Layout(width="850px"),
    )
    refresh_models_button = widgets.Button(description="Refresh models", icon="refresh")
    speaker_input = widgets.IntText(value=0, description="Speaker ID:")
    custom_checkpoint_input = widgets.Text(
        value="",
        description="Custom path:",
        placeholder="generator checkpoint file or checkpoint directory",
        disabled=True,
        layout=widgets.Layout(width="850px"),
    )
    model_output = widgets.Output()
    run_button = widgets.Button(
        description="Run vowel edit",
        button_style="primary",
        icon="play",
        disabled=True,
    )
    alignment_output = widgets.Output()
    run_output = widgets.Output()
    state = {"preview": None, "last_result": None, "models": []}

    def selected_features():
        return tuple(
            name for name in TRANSFER_FEATURE_ORDER
            if feature_checkboxes[name].value
        )

    def refresh_alignment(*_):
        with alignment_output:
            clear_output(wait=True)
            try:
                if neutral_dropdown.value == donor_dropdown.value:
                    raise ValueError("Neutral and donor recordings must be different files.")
                neutral_vowels = read_vowel_intervals(
                    AUDIO_TEXTGRID_PAIRS[neutral_dropdown.value]["textgrid"]
                )
                donor_vowels = read_vowel_intervals(
                    AUDIO_TEXTGRID_PAIRS[donor_dropdown.value]["textgrid"]
                )
                preview = match_vowel_sequences(neutral_vowels, donor_vowels)
                labels = []
                for row in preview.to_dict("records"):
                    label = (
                        f"{row['vowel_id']:02d} | {row['neutral_phone']:<4} | "
                        f"{row['word']:<12} | neutral {row['neutral_start_s']:.3f}-"
                        f"{row['neutral_end_s']:.3f}s | donor {row['donor_start_s']:.3f}-"
                        f"{row['donor_end_s']:.3f}s"
                    )
                    labels.append((label, int(row["vowel_id"])))
                vowel_select.options = labels
                vowel_select.value = tuple(value for _, value in labels)
                state["preview"] = preview
                display(preview[[
                    "vowel_id", "neutral_phone", "donor_phone", "word",
                    "neutral_start_s", "neutral_end_s",
                    "donor_start_s", "donor_end_s",
                ]])
            except Exception as error:
                state["preview"] = None
                vowel_select.options = []
                vowel_select.value = ()
                print(f"Alignment error: {error}")

    def update_feature_controls(*_):
        features = selected_features()
        run_button.disabled = not features
        taper_slider.disabled = not bool(set(features) & set(ACOUSTIC_FEATURES))
        if features:
            feature_status.value = (
                "<b>Donor features:</b> " + ", ".join(features)
                + ("<br><span style='color:#b33'>Experimental donor PPG uses hard boundaries.</span>"
                   if "ppg" in features else "")
            )
        else:
            feature_status.value = (
                "<span style='color:#b33'><b>Select at least one donor feature.</b></span>"
            )

    def update_model_controls(*_):
        mode, _ = model_dropdown.value
        custom_checkpoint_input.disabled = mode != "custom"
        if mode == "adapted":
            speaker_input.value = 0
            speaker_input.disabled = True
        else:
            speaker_input.disabled = False

    def refresh_models(_=None):
        previous = model_dropdown.value
        options = build_model_options()
        values = [value for _, value in options]
        model_dropdown.options = options
        model_dropdown.value = previous if previous in values else ("pretrained", "")
        state["models"] = discover_adapted_models()
        update_model_controls()
        with model_output:
            clear_output(wait=True)
            if state["models"]:
                display(pd.DataFrame(state["models"]))
            else:
                print("No adapted generators found. Train one or use a custom path.")

    def select_all(_):
        vowel_select.value = tuple(value for _, value in vowel_select.options)

    def clear_selection(_):
        vowel_select.value = ()

    def selected_synthesis_voice():
        mode, discovered_path = model_dropdown.value
        if mode == "pretrained":
            return speaker_input.value, ""
        if mode == "adapted":
            return 0, discovered_path
        return speaker_input.value, custom_checkpoint_input.value

    def run_clicked(_):
        with run_output:
            clear_output(wait=True)
            features = selected_features()
            if not features:
                print("Select at least one donor feature before running.")
                return
            run_button.disabled = True
            try:
                speaker_id, checkpoint_text = selected_synthesis_voice()
                state["last_result"] = run_vowel_edit(
                    neutral_stem=neutral_dropdown.value,
                    donor_stem=donor_dropdown.value,
                    selected_vowel_ids=vowel_select.value,
                    transferred_features=features,
                    taper_ms=taper_slider.value,
                    speaker_id=speaker_id,
                    checkpoint_text=checkpoint_text,
                )
            except Exception as error:
                print(f"Edit failed: {error}")
                traceback.print_exc(limit=2)
            finally:
                update_feature_controls()

    neutral_dropdown.observe(refresh_alignment, names="value")
    donor_dropdown.observe(refresh_alignment, names="value")
    for checkbox in feature_checkboxes.values():
        checkbox.observe(update_feature_controls, names="value")
    model_dropdown.observe(update_model_controls, names="value")
    select_all_button.on_click(select_all)
    clear_button.on_click(clear_selection)
    refresh_models_button.on_click(refresh_models)
    run_button.on_click(run_clicked)

    display(widgets.VBox([
        widgets.HTML("<h3>Recording selection</h3>"),
        neutral_dropdown,
        donor_dropdown,
        alignment_output,
        widgets.HTML("<h3>Vowels and donor features</h3>"),
        vowel_select,
        widgets.HBox([select_all_button, clear_button]),
        widgets.HBox(list(feature_checkboxes.values())),
        feature_status,
        taper_slider,
        widgets.HTML("<h3>Synthesis voice</h3>"),
        widgets.HBox([model_dropdown, refresh_models_button]),
        speaker_input,
        custom_checkpoint_input,
        model_output,
        run_button,
        run_output,
    ]))
    refresh_alignment()
    refresh_models()
    update_feature_controls()
    return {
        "neutral": neutral_dropdown,
        "donor": donor_dropdown,
        "vowels": vowel_select,
        "features": feature_checkboxes,
        "feature_status": feature_status,
        "taper_ms": taper_slider,
        "model": model_dropdown,
        "refresh_models": refresh_models_button,
        "speaker_id": speaker_input,
        "custom_checkpoint": custom_checkpoint_input,
        "run_button": run_button,
        "state": state,
    }


CONTROLS = build_notebook_controls()

## Reading the saved tensor bundle

The `.pt` file stores CPU tensors and primitive metadata. To inspect it later:

```python
bundle = torch.load("path/to/..._features.pt", map_location="cpu", weights_only=False)
bundle["features"].keys()
bundle["metadata"]
```

Common mistakes to avoid:

- Do not choose recordings containing different sentences; vowel matching is intentionally strict.
- Keep TextGrid timestamps aligned with the corresponding WAV after any trimming.
- Choose a discovered adapted model from the Model dropdown and click Refresh models after training. Adapted models automatically use speaker ID `0`; the custom-path option remains available.
- Very long tapers can suppress most of a short vowel; acoustic tapers are capped at half the output vowel. Experimental donor PPG always uses a hard boundary splice.
- Feature checkboxes apply to every selected vowel. Duration changes require neutral unselected trajectories to be time-resampled so all ProMoNet inputs remain aligned.
